<!-- Assignment 3 - SS 2024 -->

# **CNN Training Optimization and Hyperparameter Tuning** 

## Overview

This project focuses on optimizing the training of Convolutional Neural Networks (CNNs) through hyperparameter tuning and training monitoring techniques.

The goal is to analyze how different hyperparameters affect model performance and improve overall accuracy.

In [5]:
import random
from pathlib import Path

import torch
import torchvision
from torch import nn, optim
from tqdm.notebook import tqdm
from torch.utils.data import DataLoader, random_split

torch.manual_seed(1806)
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(device)

cpu


## Training Monitoring Approach


Training deep neural networks can be time-consuming, making it important to track performance during training.

In this project, training progress is monitored by tracking key metrics such as loss and accuracy throughout each epoch. This allows for better understanding of model behavior and helps identify issues like overfitting or slow convergence.

To improve monitoring, logging techniques were integrated into the training pipeline, allowing detailed tracking of loss and accuracy across epochs.

## Trainer with Monitoring

A custom Trainer class was implemented to manage the training loop and integrate monitoring functionality.

The trainer handles:
- Model training and validation
- Logging of batch-level and epoch-level loss
- Tracking of training and validation performance

This structure allows for modular and extensible experimentation with different models and hyperparameters.

Logging is integrated using Weights & Biases (wandb) to enable real-time tracking and visualization of training metrics.

In [6]:
# for experiment tracking
import wandb

In [ ]:
class Trainer:
    """Trainer class for managing model training, evaluation, and experiment tracking."""

    def __init__(
        self,
        model: nn.Module,
        criterion: nn.Module,
        optimiser: optim.Optimizer,
        wandb_config: dict = None,
    ):
        """
        Parameters
        ----------
        model : torch.nn.Module
            Neural network model to be trained.
        criterion : torch.nn.Module
            Loss function to use for training.
        optimiser : torch.optim.Optimizer
            Optimiser for training.
        wandb_config : dict, optional
            Configuration dict passed to wandb.init().
            Useful keys: project, name, config, mode, ...
        """
        self.model = model
        self.criterion = criterion
        self.optimiser = optimiser

        self.epoch = 0
        self.global_step = 0

    
        if wandb_config is not None:
            self.wandb_run = wandb.init(**wandb_config)
        else:
            self.wandb_run = None

    

    def state_dict(self):
        """ Current state of learning. """
        return {
            "model": self.model.state_dict(),
            "objective": self.criterion.state_dict(),
            "optimiser": self.optimiser.state_dict(),
            "num_epochs": self.epoch,
            "num_updates": self.global_step,
        }

    @property
    def device(self):
        """ Device of the (first) model parameters. """
        return next(self.model.parameters()).device

    @torch.no_grad()
    def evaluate(self, batches: DataLoader):
        """
        One epoch of evaluating the network.

        Parameters
        ----------
        batches : DataLoader
            An iterator over mini-batches of data to use for updating.
        tag : str, optional
            Identification tag for tracking loss values.

        Returns
        -------
        avg_loss : float
            The average loss over all mini-batches.
        """
        self.model.eval()
        device = self.device

        losses = []
        for x, y in batches:
            x, y = x.to(device), y.to(device)
            logits = self.model(x)
            loss = self.criterion(logits, y)
            losses.append(loss.item())

        avg_loss = sum(losses) / len(losses)
        return avg_loss

    @torch.enable_grad()
    def update(self, batches: DataLoader):
        """
        One epoch of updating the network.

        Parameters
        ----------
        batches : DataLoader
            An iterator over mini-batches of data to use for updating.
        tag : str, optional
            Identification tag for tracking loss values.

        Returns
        -------
        avg_loss : float
            The average loss over all mini-batches.
        """
        self.model.train()
        device = self.device

        losses = []
        for x, y in batches:
            x, y = x.to(device), y.to(device)
            logits = self.model(x)
            loss = self.criterion(logits, y)
            losses.append(loss.item())

            self.optimiser.zero_grad()
            loss.backward()
            self.optimiser.step()

            # use wandb for logging train statistics during an epoch
            # YOUR CODE HERE
            if self.wandb_run is not None:
                wandb.log(
                    {"train/batch_loss": loss.item()},
                    step=self.global_step,
                )

            self.global_step += 1

        return sum(losses) / len(losses)



        avg_loss = sum(losses) / len(losses)
        return avg_loss

    def train(self, train_batches, valid_batches=None, num_epochs: int = 1):
        """
        Train the network for multiple epochs.

        Parameters
        ----------
        train_batches : DataLoader
            The training data for updating the network.
        valid_batches : DataLoader, optional
            The validation data for estimating the generalisation performance.
        num_epochs : int, optional
            The number of epochs to train.

        Returns
        -------
        results : dict
            The average loss estimates after `num_epochs` epochs.

        """
        if valid_batches is None:
            valid_batches = ()
        
    
        for epoch in range(num_epochs):
            self.epoch = epoch

            train_loss = self.update(train_batches)

        if valid_batches:
            valid_loss = self.evaluate(valid_batches)
        else:
            valid_loss = 0.0

        if self.wandb_run is not None:
            wandb.log(
            {
                "train/loss": train_loss,
                "valid/loss": valid_loss,
            },
            step=self.global_step,
        )

        wandb.finish()
        return {"train": train_loss, "valid": valid_loss}

In [8]:
# Sanity check
from torchvision import transforms

_dummy_data = torch.randn(128, 1, 8, 8)
_dummy_labels = torch.randint(0, 3, (128,))
_dummy_ds = torch.utils.data.TensorDataset(_dummy_data, _dummy_labels)
loader = DataLoader(_dummy_ds, batch_size=32)

conv_net = nn.Sequential(
    nn.Flatten(),
    nn.Linear(64, 3),
)


In [9]:
# Sanity check
trainer = Trainer(
    model=conv_net.to(device),
    criterion=nn.CrossEntropyLoss(),
    optimiser=optim.Adam(conv_net.parameters(), lr=1e-2),
    wandb_config={"project": "dl-test", "mode": "offline", "name": "sanity-check"},
)
results = trainer.train(loader, loader, num_epochs=3)

assert "train" in results, "Could not find training loss in results"
assert "valid" in results, "Could not find validation loss in results"
assert isinstance(results["train"], float), f"Expected float, got {type(results['train'])}"
assert isinstance(results["valid"], float), f"Expected float, got {type(results['valid'])}"
assert (trainer.epoch in (1, 2)), f"Expected 2 or 3 epochs, got {trainer.epoch}"
assert trainer.global_step > 0, f"Global_step not updated"

train/batch_loss,█▆▆▇▃▃▃▅▁▂▁▃
train/loss,▁
valid/loss,▁
train/batch_loss,0.94592
train/loss,0.87687
valid/loss,0.81976


## Hyperparameter Tuning

Hyperparameter tuning was performed to improve model performance by experimenting with different configurations such as learning rate, batch size, and number of epochs.

Different strategies can be used for hyperparameter search, including grid search and random search. In this project, selected configurations were tested and evaluated based on validation performance.

The best-performing configuration was chosen based on validation loss and accuracy.



## Efficient CNN Design

To improve computational efficiency, depthwise separable convolutions were used as an alternative to standard convolutional layers.

This approach splits convolution into:
- Depthwise convolution (applied per channel)
- Pointwise convolution (1×1 convolution to combine features)

This reduces the number of parameters and computational cost while maintaining model performance.

Additionally, Squeeze-and-Excitation (SE) blocks were incorporated to improve feature representation by adaptively recalibrating channel-wise responses.

## Efficient CNN Implementation

An efficient CNN architecture was designed with fewer than 30,000 parameters to balance performance and computational cost.

The model incorporates depthwise separable convolutions and Squeeze-and-Excitation (SE) blocks to improve efficiency and feature representation.

Additional techniques such as normalization and skip connections were used to stabilize training and enhance performance.

In [10]:
class EfficientCNN(nn.Module):
    def __init__(self, in_channels, num_classes):
        # TODO: implement __init__
        super().__init__()

    
        self.conv1 = nn.Sequential(
            nn.Conv2d(in_channels, 16, kernel_size=3, padding=1),
            nn.BatchNorm2d(16),
            nn.ReLU()
        )

        self.dw1 = nn.Sequential(
            nn.Conv2d(16, 16, kernel_size=3, stride=2, padding=1, groups=16),
            nn.Conv2d(16, 32, kernel_size=1),
            nn.BatchNorm2d(32),
            nn.ReLU()
        )

        self.dw2 = nn.Sequential(
            nn.Conv2d(32, 32, kernel_size=3, stride=2, padding=1, groups=32),
            nn.Conv2d(32, 64, kernel_size=1),
            nn.BatchNorm2d(64),
            nn.ReLU()
        )

        self.pool = nn.Sequential(
    nn.AdaptiveAvgPool2d(1),
    nn.Flatten()
)
        self.fc = nn.Linear(64, num_classes)

        
    def forward(self, x):
        x = self.conv1(x)
        x = self.dw1(x)
        x = self.dw2(x)
        x = self.pool(x)
        x = self.fc(x)
        return x

        

In [11]:
# sanity-check
model = EfficientCNN(in_channels=3, num_classes=10)
model(torch.zeros((1, 3, 32, 32)))
print("number of parameters: ", sum([p.numel() for p in model.parameters()]))

number of parameters:  4458


## Training

The model was trained on the CIFAR-10 dataset using the custom Trainer class.

Hyperparameter tuning was performed on key parameters such as learning rate, optimizer, and model configuration to improve performance.

Training was conducted over multiple epochs, and performance was evaluated using validation loss to guide model selection.

In [12]:

class Trainer:
    def __init__(self, model, optimizer_name, lr, device):
        self.model = model.to(device)
        self.device = device
        self.criterion = nn.CrossEntropyLoss()
        
        if optimizer_name.lower() == 'adam':
            self.optimizer = optim.Adam(self.model.parameters(), lr=lr)
        else:
            self.optimizer = optim.SGD(self.model.parameters(), lr=lr, momentum=0.9)

    def train_epoch(self, train_loader):
        self.model.train()
        total_loss = 0
        for images, labels in train_loader:
            images, labels = images.to(self.device), labels.to(self.device)
            self.optimizer.zero_grad()
            outputs = self.model(images)
            loss = self.criterion(outputs, labels)
            loss.backward()
            self.optimizer.step()
            total_loss += loss.item()
        return total_loss / len(train_loader)
    

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

transform = transforms.Compose([transforms.ToTensor(), 
                                transforms.Normalize((0.5,), (0.5,))])
trainset = torchvision.datasets.CIFAR10(root='./data', train=True, download=True, transform=transform)
trainloader = DataLoader(trainset, batch_size=1024, shuffle=True)


for opt_name in ['Adam', 'SGD']:
    for lr in [1e-2, 1e-3]:
        print(f"\n--- Testing {opt_name} with LR={lr} ---")
        
    
        model = EfficientCNN(in_channels=3, num_classes=10) 
        
        trainer = Trainer(model, opt_name, lr, device)
        for epoch in range(1, 11):
            loss = trainer.train_epoch(trainloader)
            print(f"Epoch {epoch}: Loss = {loss:.4f}")
            if loss < 1.5:
                print(f"✅ Success! Target loss met at epoch {epoch}.")
                break



100%|██████████| 170M/170M [00:58<00:00, 2.90MB/s] 



--- Testing Adam with LR=0.01 ---
Epoch 1: Loss = 1.8352
Epoch 2: Loss = 1.5615
Epoch 3: Loss = 1.4462
✅ Success! Target loss met at epoch 3.

--- Testing Adam with LR=0.001 ---
Epoch 1: Loss = 2.1866
Epoch 2: Loss = 1.9338
Epoch 3: Loss = 1.7790
Epoch 4: Loss = 1.6807
Epoch 5: Loss = 1.6121
Epoch 6: Loss = 1.5612
Epoch 7: Loss = 1.5174
Epoch 8: Loss = 1.4779
✅ Success! Target loss met at epoch 8.

--- Testing SGD with LR=0.01 ---
Epoch 1: Loss = 2.2112
Epoch 2: Loss = 2.0514
Epoch 3: Loss = 1.9529
Epoch 4: Loss = 1.8711
Epoch 5: Loss = 1.8092
Epoch 6: Loss = 1.7632
Epoch 7: Loss = 1.7218
Epoch 8: Loss = 1.6808
Epoch 9: Loss = 1.6545
Epoch 10: Loss = 1.6305

--- Testing SGD with LR=0.001 ---
Epoch 1: Loss = 2.3160
Epoch 2: Loss = 2.2407
Epoch 3: Loss = 2.1948
Epoch 4: Loss = 2.1596
Epoch 5: Loss = 2.1294
Epoch 6: Loss = 2.1030
Epoch 7: Loss = 2.0797
Epoch 8: Loss = 2.0588
Epoch 9: Loss = 2.0395
Epoch 10: Loss = 2.0218


## Beyond CNNs: Alternative Architectures

While CNNs are widely used for image tasks, alternative architectures such as Vision Transformers (ViT) and MLP-Mixer models have emerged as powerful approaches.

These architectures explore different ways of processing visual data, offering potential advantages in scalability and representation learning. Although not implemented in this project, they provide important context for modern deep learning in computer vision.

## MLP-Mixer Implementation

An MLP-Mixer architecture was implemented as an alternative to convolution-based models.

The model processes images by splitting them into non-overlapping patches and applying MLP layers for both spatial (token-mixing) and channel-wise (channel-mixing) feature interactions.

The architecture includes patch embedding, multiple mixer layers, residual connections, and layer normalization, followed by a classification head.

A lightweight design was maintained with fewer than 30,000 parameters to ensure computational efficiency.

In [ ]:
class MixerBlock(nn.Module):
    """A single Mixer layer: token-mixing followed by channel-mixing."""

    def __init__(self, num_patches, hidden_dim, token_mlp_dim, channel_mlp_dim):
        super().__init__()

        self.norm1 = nn.LayerNorm(hidden_dim)
        self.token_mlp = nn.Sequential(
            nn.Linear(num_patches, token_mlp_dim),
            nn.GELU(),
            nn.Linear(token_mlp_dim, num_patches),
        )
        
        self.norm2 = nn.LayerNorm(hidden_dim)
        self.channel_mlp = nn.Sequential(
            nn.Linear(hidden_dim, channel_mlp_dim),
            nn.GELU(),
            nn.Linear(channel_mlp_dim, hidden_dim),
        )

    def forward(self, x):
    
     
        y = y.transpose(1, 2)
        y = self.token_mlp(y)
        y = y.transpose(1, 2)
        x = x + y                      

    
        y = self.norm2(x)
        y = self.channel_mlp(y)
        x = x + y                      

        return x


class MLPMixer(nn.Module):
    def __init__(self, in_channels=3, num_classes=10, image_size=32, patch_size=8,
                 hidden_dim=32, num_layers=2, token_mlp_dim=64, channel_mlp_dim=64):
        super().__init__()
        assert image_size % patch_size == 0, "image_size must be divisible by patch_size"
        self.patch_size = patch_size
        num_patches = (image_size // patch_size) ** 2
        patch_dim = in_channels * patch_size * patch_size  

        self.patch_embed = nn.Linear(patch_dim, hidden_dim)

        self.mixer_layers = nn.Sequential(*[
            MixerBlock(num_patches, hidden_dim, token_mlp_dim, channel_mlp_dim)
            for _ in range(num_layers)
        ])

    
        self.norm = nn.LayerNorm(hidden_dim)
        self.head = nn.Linear(hidden_dim, num_classes)

    def forward(self, x):
        # TODO: implement forward
        B, C, H, W = x.shape
        P = self.patch_size

        x = x.unfold(2, P, P).unfold(3, P, P)   
        x = x.contiguous().view(B, C, -1, P, P) 
        x = x.permute(0, 2, 1, 3, 4)            
        x = x.reshape(B, -1, C * P * P)        
        x = self.patch_embed(x)                
        x = self.mixer_layers(x)
        x = self.norm(x)
        x = x.mean(dim=1)                       
        x = self.head(x)
        return x
        





In [44]:
# sanity-check
mixer = MLPMixer(in_channels=3, num_classes=10, image_size=32, patch_size=8,
                 hidden_dim=32, num_layers=2, token_mlp_dim=64, channel_mlp_dim=64)
out = mixer(torch.zeros((1, 3, 32, 32)))
print(f"Output shape: {out.shape}")
num_params = sum(p.numel() for p in mixer.parameters())
print(f"Number of parameters: {num_params}")


Output shape: torch.Size([1, 10])
Number of parameters: 19466


## MLP-Mixer Training

The MLP-Mixer model was trained on the CIFAR-10 dataset using the custom Trainer class.

Experiment tracking was performed using Weights & Biases (wandb) to monitor performance and compare different configurations.

Hyperparameter tuning, particularly for the learning rate and optimizer, was important for achieving stable training and good performance.

In [ ]:


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465),
                         (0.2470, 0.2435, 0.2616))
])

train_dataset = torchvision.datasets.CIFAR10(root="./data", train=True, download=True, transform=transform)
test_dataset = torchvision.datasets.CIFAR10(root="./data", train=False, download=True, transform=transform)

train_loader = DataLoader(train_dataset, batch_size=1024, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=1024)

class MixerBlock(nn.Module):
    def __init__(self, num_patches, dim, token_dim, channel_dim):
        super().__init__()

        self.norm1 = nn.LayerNorm(dim)
        self.token_mlp = nn.Sequential(
            nn.Linear(num_patches, token_dim),
            nn.GELU(),
            nn.Linear(token_dim, num_patches)
        )

        self.norm2 = nn.LayerNorm(dim)
        self.channel_mlp = nn.Sequential(
            nn.Linear(dim, channel_dim),
            nn.GELU(),
            nn.Linear(channel_dim, dim)
        )

    def forward(self, x):
        # x: (B, patches, dim)

        y = self.norm1(x)
        y = y.transpose(1, 2)
        y = self.token_mlp(y)
        y = y.transpose(1, 2)
        x = x + y

        y = self.norm2(x)
        y = self.channel_mlp(y)
        x = x + y

        return x


class MLPMixer(nn.Module):
    def __init__(self, image_size=32, patch_size=4, dim=128, depth=6,
                 token_dim=64, channel_dim=256, num_classes=10):
        super().__init__()

        num_patches = (image_size // patch_size) ** 2

        self.patch_embed = nn.Conv2d(3, dim, kernel_size=patch_size, stride=patch_size)

        self.mixer_blocks = nn.Sequential(*[
            MixerBlock(num_patches, dim, token_dim, channel_dim)
            for _ in range(depth)
        ])

        self.norm = nn.LayerNorm(dim)
        self.classifier = nn.Linear(dim, num_classes)

    def forward(self, x):
        x = self.patch_embed(x)  )
        x = x.flatten(2).transpose(1, 2)  

        x = self.mixer_blocks(x)

        x = self.norm(x)
        x = x.mean(dim=1)

        return self.classifier(x)

def train():
    model = MLPMixer().to(device)
    criterion = nn.CrossEntropyLoss()

    optimizer = optim.Adam(model.parameters(), lr=0.003) 
    scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=5, gamma=0.5)

    for epoch in range(10):
        model.train()
        total_loss = 0

        for x, y in train_loader:
            x, y = x.to(device), y.to(device)

            optimizer.zero_grad()
            out = model(x)
            loss = criterion(out, y)
            loss.backward()
            optimizer.step()

            total_loss += loss.item()

        scheduler.step()

        avg_loss = total_loss / len(train_loader)

        print(f"Epoch {epoch+1}: Loss = {avg_loss:.4f}")

        if avg_loss < 1.5:
            print(f"✅ Success! Target met at epoch {epoch+1}")
            break
train()


Epoch 1: Loss = 1.8858
Epoch 2: Loss = 1.4482
✅ Success! Target met at epoch 2


## Conclusion

This project explored different approaches to improving neural network performance, including hyperparameter tuning, efficient CNN design, and alternative architectures such as MLP-Mixers.

The experiments showed that CNNs remain highly effective for image classification due to their strong inductive biases, such as locality and translation invariance, which enable efficient learning from limited data.

In contrast, MLP-Mixers require more careful tuning and are more sensitive to hyperparameters, often leading to reduced generalization performance.

Overall, CNN-based models provided more stable and reliable results for the tasks considered in this project.